# Spatial smoothing

This tutorial demonstrates how to generate voxel-wise cell type membership probabilities/relative masses for your single-cell-resolved spatial transcriptomics (ST) dataset. 

In this notebook, we will:
1. Use HypoMap3D's voxel store to initialize a new store
2. Assign cells to voxels and smooth membership probabilities

Requirements:
1. HypoMap3D's code base and environment. For this, follow the instructions at the root of the Git repo.
2. HypoMap3D's voxel store. Download it and place it into DATADIR/9_objects/spatial/hypomap3d_spatial_voxel_level.zarr
3. An ST dataset with CCF coordinates as well as HypoMap3D cell type labels as transferred in the ProjectAndTransferLabels notebook, including the filtering to cells with > 100 counts in HypoMap3D HVGs. In this tutorial, we will use HypoMap3D's cell-level spatial data. If you want to rerun the notebook as is, download the cell-level spatial data into DATADIR/9_objects/spatial/hypomap3d_spatial_cell_level.h5ad

Of course, the above paths can be modified but then they have to be changed accordingly below as well.

In [56]:
import warnings
import os
from pathlib import Path

import pandas as pd
import scanpy as sc
import numpy as np
import zarr
from zarr.core.dtype import VariableLengthUTF8
from dotenv import dotenv_values
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter
import tqdm

warnings.simplefilter(action='ignore', category=FutureWarning)
sc.set_figure_params(frameon=False, dpi=120, figsize=(5, 5))

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env")
DATADIR = Path(CONFIG['DATADIR'])

VOXEL_IN_FP = DATADIR / "9_objects" / "spatial" / 'hypomap3d_spatial_voxel_level.zarr' # for default structure
SC_IN_FP = DATADIR / "9_objects" / "spatial" / 'hypomap3d_spatial_cell_level.h5ad'

In [3]:
# Notebook constants
ROOT_REGION_ID = 997

MIN_SUPPORT = 0.15

## Init new voxel store

For this matter, we will set up a new voxel store in the same form as the HypoMap3D voxel space by copying over the groups that remain constant.

But before we do so, please provide a path for it.

In [ ]:
VOXEL_OUT_FP = DATADIR / "example_notebooks" / "SpatialSmoothing" / "voxel_store.zarr" # change it if desired

In [9]:
def copy_group(src_group, dst_group):
    # Copy arrays
    for name, arr in src_group.arrays():
        dst_group.create_array(
            name,
            data=arr,
            overwrite=True
        )
        dst_group[name].attrs.update(arr.attrs)

    for name, subgroup in src_group.groups():
        new_group = dst_group.create_group(name)
        new_group.attrs.update(subgroup.attrs)
        copy_group(subgroup, new_group)


In [7]:
# Open source store
src_store = zarr.open(VOXEL_IN_FP, mode="r")

In [13]:
# Create destination store
voxel_store = zarr.open(VOXEL_OUT_FP, mode="w")
# Copy subgroups
for group_name in ["bbox", "region"]:
    new_group = voxel_store.create_group(group_name, overwrite=True)
    new_group.attrs.update(src_store[group_name].attrs)
    copy_group(src_store[group_name], new_group)

In [14]:
voxel_store.tree()

/
├── bbox
│   ├── ccf_mm
│   │   ├── max (3,) float32
│   │   └── min (3,) float32
│   ├── shape (3,) float32
│   └── voxel_25_um
│       ├── max (3,) float32
│       └── min (3,) float32
└── region
    ├── acronyms (176,) StringDType()
    ├── ids (176,) int32
    ├── region_labels (168, 144, 107) uint32
    └── region_masks (176, 168, 144, 107) bool

## Load and prepare cell-level ST data

### Load adata

In [40]:
# Load cell-level spatial dataset
adata = sc.read_h5ad(SC_IN_FP)
adata

AnnData object with n_obs × n_vars = 1401313 × 56939
    obs: 'Dataset', 'Sample', 'Author', 'Technology', 'Donor', 'Sex', 'RegionID', 'RegionAcronym', 'RegionName', 'total_counts', 'total_hvg_counts', 'total_genes', 'IsHY', 'L1_descr_max_p', 'L1_abc_like_max_p', 'L1_max_p', 'L2_descr_max_p', 'L2_abc_like_max_p', 'L2_max_p', 'L3_descr_max_p', 'L3_abc_like_max_p', 'L3_max_p', 'L4_descr_max_p', 'L4_abc_like_max_p', 'L4_max_p', 'L5_descr_max_p', 'L5_abc_like_max_p', 'L5_max_p'
    var: 'gene_stable_ids', 'IsHVG'
    uns: 'L1_ID', 'L1_abc_like', 'L1_descr', 'L2_ID', 'L2_abc_like', 'L2_descr', 'L3_ID', 'L3_abc_like', 'L3_descr', 'L4_ID', 'L4_abc_like', 'L4_descr', 'L5_ID', 'L5_abc_like', 'L5_descr', 'flags', 'pathmat'
    obsm: 'P_L1', 'P_L2', 'P_L3', 'P_L4', 'P_L5', 'X_embed', 'X_umap', 'X_umap_sc', 'coords_ccf_mm', 'coords_ccf_mm_wrap', 'coords_source', 'coords_voxels_25um', 'coords_voxels_25um_bounded'
    varm: 'IsCaptured'

### Putting CCF coords into the right place

For purpose of guiding you through some steps you probably have to do with your data we insert the CCF coordinates back into .obs here as this is likely how your data looks

In [41]:
adata.obs[['x_ccf', 'y_ccf', 'z_ccf']] = adata.obsm['coords_ccf_mm'].copy()

This is probably how your data looks:

In [42]:
adata.obs[['x_ccf', 'y_ccf', 'z_ccf']].head()

,x_ccf,y_ccf,z_ccf
SP2-01_C57BL6J-638850.30_1018093344101520397-3,8.498350,6.113867,4.970232
SP2-01_C57BL6J-638850.30_1018093344101520363-3,8.498876,6.100787,4.942921
SP2-01_C57BL6J-638850.30_1018093344101070149-6,8.485158,5.753897,3.276683
SP2-01_C57BL6J-638850.30_1018093344101070178-4,8.494820,5.671684,3.321470
SP2-01_C57BL6J-638850.30_1018093344101520467-4,8.497904,6.138259,5.039371


__Note that we use CCF coordinates in mm!__ 

If your data comes in micrometer the next cell would most likely detect it and transform the data into mm

In [43]:
if adata.obs.x_ccf.max() > 1000:
    print("Data is most likely micrometer, transforming to mm")
    adata.obs.x_ccf /= 1000
    adata.obs.y_ccf /= 1000
    adata.obs.z_ccf /= 1000

Now let us place your ccf coordinates in mm into .obsm

In [44]:
ccf_df = adata.obs[['x_ccf', 'y_ccf', 'z_ccf']].copy()
ccf_df.columns = ['x', 'y', 'z']
adata.obsm['coords_ccf_mm'] = ccf_df

### Wrap all cells into the same hemisphere

HypoMap3D operates on one hemisphere only as to create a volume that is sampled as dense as possible.

Hence, cells protruding into the right hemisphere are mirrored into the left along the midline of the brain, which is at CCF 5.7. 

That coordiante is saved in the bounding box group of the initialized voxel store along with the other coordinates required to restrict the cell-level data to the same 3D space as HypoMap3D:

In [45]:
x_min, y_min, z_min = voxel_store["bbox"]["ccf_mm"]["min"][:]
x_max, y_max, z_max = voxel_store["bbox"]["ccf_mm"]["max"][:]

In [46]:
z_midline = z_max
z_midline

np.float32(5.7)

In [47]:
# Wrap z coordinate
z_coord = adata.obsm['coords_ccf_mm']['z'].values
z_coord_wrapped = z_midline - np.abs(z_midline - z_coord)
adata.obsm['coords_ccf_mm_wrap'] = adata.obsm['coords_ccf_mm'].copy()
adata.obsm['coords_ccf_mm_wrap']['z'] = z_coord_wrapped

### Filter cells for the bounding box of HypoMap3D's voxel space 

Using the min and max coordinates we extracted from the voxel store previously, we remove cells not within this volume.

In [ ]:
keep_mask = (
    adata.obsm['coords_ccf_mm_wrap']['x'] <= x_max) & (
    adata.obsm['coords_ccf_mm_wrap']['x'] >= x_min) & (
    adata.obsm['coords_ccf_mm_wrap']['y'] <= y_max) & (
    adata.obsm['coords_ccf_mm_wrap']['y'] >= y_min) & (
    adata.obsm['coords_ccf_mm_wrap']['z'] <= z_max) & (
    adata.obsm['coords_ccf_mm_wrap']['z'] >= z_min
    )

In [49]:
adata = adata[keep_mask, :]

### Create voxel coordinate system

This transforms mm into indices that are used to access the 25 um cubed voxel space

In [52]:
# These will be indices along the whole CCF 25 um voxel space
adata.obsm['coords_voxels_25um'] = np.floor(adata.obsm['coords_ccf_mm_wrap'] * 1000 / 25).astype(int)

# These will be indices starting at 0 at the start of HypoMap3D's voxel space
bbox_min_25um = voxel_store["bbox"]["voxel_25_um"]["min"][:]
adata.obsm['coords_voxels_25um_bounded'] = (adata.obsm['coords_voxels_25um'] - bbox_min_25um).astype(int)

### Get ID to abc-like and ID to descriptive labels mapper for later

These are used as to add more metadata into the voxel store

In [54]:
all_keys = []
all_descr = []
all_abc = []
for i in range(1, 6):
    keys = list(adata.uns[f"L{i}_ID"])
    descr = list(adata.uns[f"L{i}_descr"])
    abc = list(adata.uns[f"L{i}_abc_like"])

    all_keys.extend(keys)
    all_descr.extend(descr)
    all_abc.extend(abc)

descr_mapper = {k: v for k, v in zip(all_keys, all_descr)}
abclike_mapper = {k: v for k, v in zip(all_keys, all_abc)}

## Smoothing

In essence, the following operations are performed:

- Aggregation of cells into voxels
- Generation of support and evidence matrices for normalized convolution
- Normalized gaussian convolution of path propbabilites to get P(leaf cluster), in this case L5
- Summation of L5 descendants to derive lower granularity probailities (i.e L1-L4)

For more information, please consider the manuscript.

Please note, that by default we use a sigma of 2 (= 50um) for the smoothing operation. If you wish, you may also alter this value: Lower sigma values will result in grids that were smoothed less smoothed.

In [ ]:
SIGMA = 2

In [55]:
# Extract root mask and derive voxel shape from it
idx = list(src_store['region']['ids'][:]).index(ROOT_REGION_ID) # root region (all region encompassing)
root_mask = src_store['region']['region_masks'][idx, ...]

nx, ny, nz = root_mask.shape
n_voxels = nx * ny * nz

print((nx, ny, nz), n_voxels)

(168, 144, 107) 2588544


In [64]:
def smooth_adata(adata, voxel_store, batch_key="Sample"):

    ##################################
    ################# Sample-wise voxel aggregation
    ##################################
    print("Sample-wise voxel agg.")
    nc = adata.obsm['P_L5'].shape[1]

    # Intialize empty total grids
    total_mass = np.zeros((nc, n_voxels), dtype=np.float32)
    total_counts = np.zeros(n_voxels, dtype=np.int32)
    total_coverage = np.zeros(n_voxels, dtype=np.int32)
    total_embed = np.zeros((45, n_voxels), dtype=np.float32) # embedding are dense

    unique_sample = adata.obs[batch_key].dropna().unique()
    for sample in tqdm.tqdm(unique_sample):

        sample_indices = np.where(adata.obs[batch_key] == sample)[0]
        n_points = len(sample_indices)

        # Subset and extract only voxel coords for that sample
        subset = adata[sample_indices]
        coords = subset.obsm['coords_voxels_25um_bounded'].values
        ravel_coords = np.ravel_multi_index(coords.T, dims=(nx, ny, nz))

        # Construct counts grid
        sample_counts = np.bincount(
            ravel_coords,
            minlength=n_voxels
        ).astype(np.int32)

        # Infer coverage grid from counts grid
        sample_coverage = (sample_counts > 0).astype(np.int32)

        # Add intermediate grid to totals
        total_counts += sample_counts
        total_coverage += sample_coverage

        # Add probabilties to grid
        P = subset.obsm["P_L5"].tocoo(copy=False)
        np.add.at(
            total_mass,
            (P.col, ravel_coords[P.row]),
            P.data,
        )

        # Add embedding direct to total
        embed = np.asarray(subset.obsm["X_embed"])
        np.add.at(total_embed.T, ravel_coords, embed)

    ##################################
    ################# Setting up support matrix
    ##################################
    print("Setting up support matrix")
    sampled_mask = (total_coverage > 0).astype(np.float32) # binary matrix 1 - point was sampled in this voxel 0 - missing
    sampled_mask = sampled_mask.reshape(nx, ny, nz)

    smoothed_support = gaussian_filter(sampled_mask, sigma=SIGMA, mode='constant', cval=0)
    smoothed_root_mask = gaussian_filter(root_mask.astype(np.float32), sigma=SIGMA, mode='constant', cval=0)

    # Set OOB to nan to propagate to all further grids
    smoothed_support[~root_mask] = np.nan

    # Normalized convolution - we don't care for voxels out of bound (not in brain/root)
    smoothed_evidence = smoothed_support / (smoothed_root_mask + 1e-8)

    # Save to zarr voxel store
    voxel_store.create_array('evidence', shape=(nx, ny, nz), dtype=np.float32, overwrite=True, chunks=(nx, ny, nz))
    voxel_store['evidence'] = smoothed_evidence

    ##################################
    ################# Smoothing grids
    ##################################
    print("Smoothing grids")
    # normalization_factors
    inv_total_coverage = total_coverage.astype(np.float32, copy=True)
    inv_total_coverage[inv_total_coverage > 0] = 1 / inv_total_coverage[inv_total_coverage > 0]

    # ---------------------------------- Counts ---------------------------------- #

    # Compute average counts per voxel across all samples
    avg_counts = total_counts * inv_total_coverage
    avg_counts = avg_counts.reshape(nx, ny, nz)

    # Smooth counts
    smoothed_avg_counts = gaussian_filter(avg_counts, sigma=SIGMA, mode='constant', cval=0)

    # Normalize convolution by support grid
    smoothed_avg_counts /= smoothed_support

    # Save to zarr voxel store
    voxel_store.create_array('counts', shape=(nx, ny, nz), dtype=np.float32, overwrite=True, chunks=(nx, ny, nz))
    voxel_store['counts'] = smoothed_avg_counts.astype(np.float32)

    # --------------------------------- Embedding -------------------------------- #

    # Compute average embedding per voxel across all samples
    avg_embed = total_embed * inv_total_coverage
    avg_embed = avg_embed.reshape(45, nx, ny, nz)

    n_features = total_embed.shape[0]

    # Smooth embeddings
    smoothed_avg_embed = gaussian_filter(avg_embed, sigma=SIGMA, mode='constant', cval=0, axes=(1, 2, 3))

    # Normalize convolution by support grid
    smoothed_avg_embed /= smoothed_support

    # Save to zarr voxel store
    voxel_store.create_array('latent_embedding', shape=(n_features, nx, ny, nz), dtype=np.float32, overwrite=True, chunks=(1, nx, ny, nz))
    voxel_store['latent_embedding'] = smoothed_avg_embed

    # ----------------------------------- Mass ----------------------------------- #
    # All operations are in-place to avoid copies in memory

    # Compute average mass
    avg_mass = total_mass
    np.multiply(avg_mass, inv_total_coverage, out=avg_mass)
    avg_mass = avg_mass.reshape(nc, nx, ny, nz)

    # Smoothing along x, y and z dimensions
    smoothed_avg_mass = avg_mass
    gaussian_filter(
        smoothed_avg_mass,
        sigma=SIGMA,
        mode="constant",
        cval=0,
        axes=(1, 2, 3),
        output=smoothed_avg_mass,
    )

    # Normalize convolution by support
    smoothed_avg_mass /= smoothed_support[None, :, :, :]

    # ------------------------------- Probabilities ------------------------------ #

    # Probabilities theoretically can be derived by mass / counts
    smoothed_probs = smoothed_avg_mass / smoothed_avg_counts[None, ...].astype(np.float32)

    # Renormalize by voxelsums for true probabilities
    voxel_sum = np.sum(smoothed_probs, axis=0)
    smoothed_probs /= voxel_sum[None, :, :, :]

    # --------------------- Propagating Upwards to L1, L2 etc -------------------- #

    pathmat = adata.uns['pathmat']

    for i, level in enumerate(pathmat.T):

        if i == (pathmat.shape[1] - 1):
            labels = level
            n_features = len(labels)

            P = smoothed_probs
            M = smoothed_avg_mass

        else:
            labels, inv = np.unique(level, return_inverse=True)
            n_features = len(labels)

            A = inv[:, None] == np.arange(len(labels))

            P = np.tensordot(A.T, smoothed_probs, axes=(1, 0))
            M = np.tensordot(A.T, smoothed_avg_mass, axes=(1, 0))

        key = f"L{i+1}"

        # Get relative mass safely
        mass_sum = np.nansum(M, axis=(1, 2, 3), keepdims=True)  # (n_features, 1, 1, 1)
        RM = np.where(
            mass_sum > 0,
            M / np.where(mass_sum > 0, mass_sum, 1.0),
            0.0
        ).astype(np.float32)

        # Create level group
        lvl_grp = voxel_store.create_group(key, overwrite=True)

        # Initialise arrays in Zarr voxel store
        lvl_grp.create_array('prob', shape=(n_features, nx, ny, nz), dtype=np.float32, overwrite=True, chunks=(1, nx, ny, nz))
        lvl_grp.create_array('mass', shape=(n_features, nx, ny, nz), dtype=np.float32, overwrite=True, chunks=(1, nx, ny, nz))
        lvl_grp.create_array('rel_mass', shape=(n_features, nx, ny, nz), dtype=np.float32, overwrite=True, chunks=(1, nx, ny, nz))

        # Write to Zarr arrays
        lvl_grp['prob'] = P
        lvl_grp["mass"] = M
        lvl_grp["rel_mass"] = RM

        # Write metadata
        lvl_grp.create_array(
            "ids",
            shape=(n_features,),
            dtype=VariableLengthUTF8(),
            overwrite=True,
        )
        lvl_grp["ids"][:] = labels

        lvl_grp.create_array(
            "descr",
            shape=(n_features,),
            dtype=VariableLengthUTF8(),
            overwrite=True,
        )
        lvl_grp["descr"][:] = [descr_mapper[i] for i in labels]

        lvl_grp.create_array(
            "abc-like",
            shape=(n_features,),
            dtype=VariableLengthUTF8(),
            overwrite=True,
        )
        lvl_grp["abc-like"][:] = [abclike_mapper[i] for i in labels]

In [66]:
smooth_adata(adata, voxel_store, batch_key="Sample") # batch key is the .obs column of your sample information

Sample-wise voxel agg.


100%|█████████████████████████████████████████| 158/158 [01:38<00:00,  1.60it/s]


Setting up support matrix
Smoothing grids


In [67]:
voxel_store.tree()

/
├── L1
│   ├── abc-like (6,) StringDType()
│   ├── descr (6,) StringDType()
│   ├── ids (6,) StringDType()
│   ├── mass (6, 168, 144, 107) float32
│   ├── prob (6, 168, 144, 107) float32
│   └── rel_mass (6, 168, 144, 107) float32
├── L2
│   ├── abc-like (82,) StringDType()
│   ├── descr (82,) StringDType()
│   ├── ids (82,) StringDType()
│   ├── mass (82, 168, 144, 107) float32
│   ├── prob (82, 168, 144, 107) float32
│   └── rel_mass (82, 168, 144, 107) float32
├── L3
│   ├── abc-like (274,) StringDType()
│   ├── descr (274,) StringDType()
│   ├── ids (274,) StringDType()
│   ├── mass (274, 168, 144, 107) float32
│   ├── prob (274, 168, 144, 107) float32
│   └── rel_mass (274, 168, 144, 107) float32
├── L4
│   ├── abc-like (716,) StringDType()
│   ├── descr (716,) StringDType()
│   ├── ids (716,) StringDType()
│   ├── mass (716, 168, 144, 107) float32
│   ├── prob (716, 168, 144, 107) float32
│   └── rel_mass (716, 168, 144, 107) float32
├── L5
│   ├── abc-like (932,) StringDType()
│   ├── descr (932,) StringDType()
│   ├── ids (932,) StringDType()
│   ├── mass (932, 168, 144, 107) float32
│   ├── prob (932, 168, 144, 107) float32
│   └── rel_mass (932, 168, 144, 107) float32
├── bbox
│   ├── ccf_mm
│   │   ├── max (3,) float32
│   │   └── min (3,) float32
│   ├── shape (3,) float32
│   └── voxel_25_um
│       ├── max (3,) float32
│       └── min (3,) float32
├── counts (168, 144, 107) float32
├── evidence (168, 144, 107) float32
├── latent_embedding (45, 168, 144, 107) float32
└── region
    ├── acronyms (176,) StringDType()
    ├── ids (176,) int32
    ├── region_labels (168, 144, 107) uint32
    └── region_masks (176, 168, 144, 107) bool

To see how to interact with the voxel store please check out the sp_slice tutorial! 